In [116]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import gpytorch
import torch
# with pd.ExcelFile("datasets/stresstool_update.xls") as xls:
#     df0 = pd.read_excel(xls, "UserInterface")
#     df1 = pd.read_excel(xls, "Kriging")
#     df2 = pd.read_excel(xls, "Quadratic")

In [117]:
df_data = pd.read_csv("datasets/Kriging_data.csv", header=[0, 1])
df_data.columns

MultiIndex([(        't_lf', 'Unnamed: 0_level_1'),
            (    't_solder', 'Unnamed: 1_level_1'),
            ('MidDieStress',               'top2'),
            ('MidDieStress',               'top5'),
            ('MidDieStress',               'bot4'),
            ('MidDieStress',               'bot5'),
            ('MaxDieStress',               'top2'),
            ('MaxDieStress',               'top5'),
            ('MaxDieStress',               'bot4'),
            ('MaxDieStress',               'bot5')],
           )

In [118]:
X_bounds = np.array([[.2, .8], [0., 1.]])

In [119]:
X = df_data[["t_lf", "t_solder"]].values
X

array([[0.257 , 0.054 ],
       [0.771 , 0.02  ],
       [0.2   , 0.037 ],
       [0.543 , 0.076 ],
       [0.657 , 0.046 ],
       [0.429 , 0.041 ],
       [0.943 , 0.033 ],
       [0.886 , 0.05  ],
       [1.    , 0.067 ],
       [0.314 , 0.071 ],
       [0.6   , 0.029 ],
       [0.829 , 0.08  ],
       [0.714 , 0.063 ],
       [0.371 , 0.024 ],
       [0.486 , 0.059 ],
       [0.2842, 0.0263],
       [0.5789, 0.0516],
       [0.7474, 0.0389],
       [0.9158, 0.0611],
       [0.4526, 0.0737],
       [0.967 , 0.023 ],
       [0.233 , 0.065 ],
       [0.633 , 0.078 ],
       [0.8   , 0.028 ],
       [0.333 , 0.047 ]])

In [120]:
X_scaled = torch.tensor((X - X_bounds[:, 0]) / (X_bounds[:, 1] - X_bounds[:, 0]))
X_scaled

tensor([[0.0950, 0.0540],
        [0.9517, 0.0200],
        [0.0000, 0.0370],
        [0.5717, 0.0760],
        [0.7617, 0.0460],
        [0.3817, 0.0410],
        [1.2383, 0.0330],
        [1.1433, 0.0500],
        [1.3333, 0.0670],
        [0.1900, 0.0710],
        [0.6667, 0.0290],
        [1.0483, 0.0800],
        [0.8567, 0.0630],
        [0.2850, 0.0240],
        [0.4767, 0.0590],
        [0.1403, 0.0263],
        [0.6315, 0.0516],
        [0.9123, 0.0389],
        [1.1930, 0.0611],
        [0.4210, 0.0737],
        [1.2783, 0.0230],
        [0.0550, 0.0650],
        [0.7217, 0.0780],
        [1.0000, 0.0280],
        [0.2217, 0.0470]], dtype=torch.float64)

In [121]:
y = df_data[('MidDieStress', 'top2')].values[:, None]
y

array([[-23926.825022],
       [  8673.64329 ],
       [  -550.882608],
       [ -2899.288678],
       [  2946.433399],
       [ -7561.378932],
       [ -3922.703139],
       [  -608.634532],
       [  -655.532497],
       [-12983.097303],
       [ -8245.748616],
       [  -463.168584],
       [  -753.04254 ],
       [ -6011.239726],
       [  -885.012646],
       [ 12241.338842],
       [  9311.823487],
       [ -5407.810277],
       [  2485.45288 ],
       [  9408.70384 ],
       [  2507.390278],
       [ 17864.950155],
       [   487.438467],
       [ -2905.100404],
       [ 11766.761307]])

In [122]:
scaler = StandardScaler()
y_scaled = torch.tensor(scaler.fit_transform(y))
y_scaled

tensor([[-2.7842],
        [ 1.0098],
        [-0.0637],
        [-0.3370],
        [ 0.3433],
        [-0.8796],
        [-0.4561],
        [-0.0704],
        [-0.0759],
        [-1.5106],
        [-0.9592],
        [-0.0535],
        [-0.0872],
        [-0.6992],
        [-0.1026],
        [ 1.4250],
        [ 1.0841],
        [-0.6290],
        [ 0.2897],
        [ 1.0954],
        [ 0.2922],
        [ 2.0795],
        [ 0.0571],
        [-0.3377],
        [ 1.3698]], dtype=torch.float64)

In [123]:
# We will use the simplest form of GP model, exact inference
class ExactGPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super(ExactGPModel, self).__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.ZeroMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(gpytorch.kernels.RBFKernel())

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

# initialize likelihood and model
likelihood = gpytorch.likelihoods.GaussianLikelihood()
model = ExactGPModel(X_scaled, y_scaled, likelihood)

In [124]:
training_iter = 50

# Find optimal model hyperparameters
model.train()
likelihood.train()

# Use the adam optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)  # Includes GaussianLikelihood parameters

# "Loss" for GPs - the marginal log likelihood
mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

for i in range(training_iter):
    # Zero gradients from previous iteration
    optimizer.zero_grad()
    # Output from model
    output = model(X_scaled)
    # Calc loss and backprop gradients
    loss = -mll(output, y_scaled.flatten())
    loss.backward()
    print('Iter %d/%d - Loss: %.3f   lengthscale: %.3f   noise: %.3f' % (
        i + 1, training_iter, loss.item(),
        model.covar_module.base_kernel.lengthscale.item(),
        model.likelihood.noise.item()
    ))
    optimizer.step()

Iter 1/50 - Loss: 1.565   lengthscale: 0.693   noise: 0.693
Iter 2/50 - Loss: 1.542   lengthscale: 0.744   noise: 0.744
Iter 3/50 - Loss: 1.524   lengthscale: 0.798   noise: 0.798
Iter 4/50 - Loss: 1.510   lengthscale: 0.853   noise: 0.852
Iter 5/50 - Loss: 1.499   lengthscale: 0.909   noise: 0.906
Iter 6/50 - Loss: 1.491   lengthscale: 0.967   noise: 0.959
Iter 7/50 - Loss: 1.485   lengthscale: 1.025   noise: 1.009
Iter 8/50 - Loss: 1.481   lengthscale: 1.083   noise: 1.057
Iter 9/50 - Loss: 1.478   lengthscale: 1.142   noise: 1.100
Iter 10/50 - Loss: 1.476   lengthscale: 1.200   noise: 1.138
Iter 11/50 - Loss: 1.474   lengthscale: 1.257   noise: 1.170
Iter 12/50 - Loss: 1.473   lengthscale: 1.313   noise: 1.195
Iter 13/50 - Loss: 1.472   lengthscale: 1.367   noise: 1.214
Iter 14/50 - Loss: 1.470   lengthscale: 1.420   noise: 1.227
Iter 15/50 - Loss: 1.468   lengthscale: 1.472   noise: 1.235
Iter 16/50 - Loss: 1.467   lengthscale: 1.521   noise: 1.236
Iter 17/50 - Loss: 1.465   length

In [125]:
x_grid = torch.linspace(0, 1, 50)
test_x = torch.stack(torch.meshgrid(x_grid, x_grid)).reshape(2, -1).T

In [126]:
# Get into evaluation (predictive posterior) mode
model.eval()
likelihood.eval()
# Test points are regularly spaced along [0,1]
# Make predictions by feeding model through likelihood
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    observed_pred = likelihood(model(test_x))

RuntimeError: Flattening the training labels failed. The most common cause of this error is that the shapes of the prior mean and the training labels are mismatched. The shape of the train targets is torch.Size([25, 1]), while the reported shape of the mean is torch.Size([25]).